In [123]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, f1_score, fbeta_score
from sklearn.neighbors import BallTree
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [124]:
training_df = pd.read_csv(
    filepath_or_buffer='training_faults_diagnostics.csv',
    low_memory=False
)
training_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 543484 entries, 0 to 543483
Data columns (total 47 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   RecordID                   543484 non-null  int64  
 1   EventTimeStamp             543484 non-null  object 
 2   eventDescription           511944 non-null  object 
 3   ecuSoftwareVersion         430359 non-null  object 
 4   ecuModel                   515906 non-null  object 
 5   ecuMake                    515906 non-null  object 
 6   ecuSource                  543484 non-null  int64  
 7   spn                        543484 non-null  int64  
 8   fmi                        543484 non-null  int64  
 9   active                     543484 non-null  bool   
 10  activeTransitionCount      543484 non-null  int64  
 11  EquipmentID                543484 non-null  object 
 12  MCTNumber                  543484 non-null  int64  
 13  Latitude                   54

In [125]:
# Manually split training, validation, and testing data by timestamp
training_df = training_df.sort_values(by='EventTimeStamp').reset_index(drop=True)

In [126]:
# Conver SPN and FMI values to strings
training_df['spn'] = training_df['spn'].astype(str)
training_df['fmi'] = training_df['fmi'].astype(str)

In [127]:
target = 'Derate_Target_12.0-2.0'

In [128]:
features = [
    'spn',
    'fmi',
    'Severity_Level',
    'BarometricPressure',
    'EngineCoolantTemperature',
    'EngineLoad',
    'EngineOilPressure',
    'EngineOilTemperature',
    'EngineRpm',
    'FuelRate',
    'FuelTemperature',
    'IntakeManifoldTemperature',
    'Speed',
    'SwitchedBatteryVoltage',
    'Throttle',
    'TurboBoostPressure'
]
len(features)

16

In [129]:
# Create dataset with desired features
X = training_df[features]
y = training_df[target]

## Identify features for imputing missing values

In [130]:
# Group categorical columns
categorical_columns = X.select_dtypes(include=["object", "bool"]).columns
print(categorical_columns)
print(len(categorical_columns))

# Group numeric columns
numeric_columns = X.select_dtypes(include=["int64", "float64"]).columns
print(numeric_columns)
print(len(numeric_columns))

Index(['spn', 'fmi', 'Severity_Level'], dtype='object')
3
Index(['BarometricPressure', 'EngineCoolantTemperature', 'EngineLoad',
       'EngineOilPressure', 'EngineOilTemperature', 'EngineRpm', 'FuelRate',
       'FuelTemperature', 'IntakeManifoldTemperature', 'Speed',
       'SwitchedBatteryVoltage', 'Throttle', 'TurboBoostPressure'],
      dtype='object')
13


In [131]:
# Group numeric columns by threshold
nan_low_threshold = 0.4

low_nan_numeric_columns = X[numeric_columns].columns[
    X[numeric_columns].isna().mean() <= nan_low_threshold
]
print(low_nan_numeric_columns)
print(len(low_nan_numeric_columns))

medium_nan_numeric_columns = X[numeric_columns].columns[
    X[numeric_columns].isna().mean() > nan_low_threshold
]
print(medium_nan_numeric_columns)
print(len(medium_nan_numeric_columns))

Index(['BarometricPressure', 'EngineCoolantTemperature', 'EngineLoad',
       'EngineOilPressure', 'EngineOilTemperature', 'EngineRpm', 'FuelRate',
       'IntakeManifoldTemperature', 'Speed', 'Throttle', 'TurboBoostPressure'],
      dtype='object')
11
Index(['FuelTemperature', 'SwitchedBatteryVoltage'], dtype='object')
2


## Split training dataset

In [132]:
total_rows = X.shape[0]
training_rows = int(total_rows * 0.6)
validation_rows = int(total_rows * 0.2)
testing_rows = total_rows - training_rows - validation_rows

print(f'60% Training: {training_rows}')
print(f'20% Validation: {validation_rows}')
print(f'20% Testing: {testing_rows}')

60% Training: 326090
20% Validation: 108696
20% Testing: 108698


In [133]:
num_training_rows = training_rows
num_validation_rows = validation_rows

X_train = X.iloc[:num_training_rows]
y_train = y.iloc[:num_training_rows]

X_val = X.iloc[num_training_rows : num_training_rows + num_validation_rows]
y_val = y.iloc[num_training_rows : num_training_rows + num_validation_rows]

X_test = X.iloc[num_training_rows + num_validation_rows :]
y_test = y.iloc[num_training_rows + num_validation_rows :]

In [134]:
# Verify split has the same number of rows as before the split
print(X_train.shape[0] + X_val.shape[0] + X_test.shape[0])
print(training_df.shape[0])

543484
543484


## Create pipeline and fit model

In [135]:
categorical_pipe = Pipeline(
    steps=[
        ('categorical_imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore'))
    ]
)

low_nan_numeric_pipe = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('low_nan_numeric_imputer', SimpleImputer(strategy='median'))
    ]
)

medium_nan_numeric_pipe = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('medium_nan_numeric_imputer', IterativeImputer(max_iter=20, random_state=30))
    ]
)

In [136]:
ct = ColumnTransformer(
    transformers=[
        ('categorical_pipe', categorical_pipe, categorical_columns),
        ('low_nan_numeric_pipe', low_nan_numeric_pipe, low_nan_numeric_columns),
        ('medium_nan_numeric_pipe', medium_nan_numeric_pipe, medium_nan_numeric_columns)
    ]
)

In [137]:
pipe = Pipeline(
    steps=[
        ('transformer', ct),
        ('model', MLPClassifier(
            activation='relu',
            hidden_layer_sizes=(32,32,32)
        ))
    ]
)

In [138]:
pipe.fit(X_train, y_train)

Pipeline(steps=[('transformer',
                 ColumnTransformer(transformers=[('categorical_pipe',
                                                  Pipeline(steps=[('categorical_imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['spn', 'fmi', 'Severity_Level'], dtype='object')),
                                                 ('low_nan_numeric_pipe',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler()),
                                                                  ('low_nan_numeri...
       'IntakeManifoldTemperature', 'Speed', 'Throttle', 'TurboBoostPressure'],
      dtype='object')),
                                                 ('medium_nan_numeric_pipe',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler()),
                                                                  ('medium_nan_numeric_imputer',
                                                                   IterativeImputer(max_iter=20,
                                                                                    random_state=30))]),
                                                  Index(['FuelTemperature', 'SwitchedBatteryVoltage'], dtype='object'))])),
                ('model', MLPClassifier(hidden_layer_sizes=(32, 32, 32)))])

## Adjust Threshold

In [156]:
y_val_pred_proba = pipe.predict_proba(X_val)[:,1]

In [157]:
candidate_thresholds = np.arange(start = 0.01, stop = 0.925, step = 0.01)
thresholds = pd.DataFrame({'threshold': candidate_thresholds})
thresholds['f1'] = thresholds['threshold'].apply(lambda x: f1_score(y_val, y_val_pred_proba > x))
thresholds.sort_values('f1', ascending = False).head()

,threshold,f1
3,0.04,0.084548
4,0.05,0.082902
2,0.03,0.080292
6,0.07,0.071279
5,0.06,0.069767


In [172]:
beta = 0.5

thresholds = pd.DataFrame({'threshold': candidate_thresholds})
thresholds['fbeta'] = thresholds['threshold'].apply(lambda x: fbeta_score(y_val, y_val_pred_proba > x, beta = beta))
thresholds.sort_values('fbeta', ascending = False).head()

,threshold,fbeta
26,0.27,0.072464
25,0.26,0.071869
24,0.25,0.067437
23,0.24,0.065913
33,0.34,0.063939


In [193]:
threshold = 0.5

y_pred_proba_train = pipe.predict_proba(X_train)[:,1]
y_pred_proba_test = pipe.predict_proba(X_test)[:,1]

y_pred_train = y_pred_proba_train > threshold
y_pred_test = y_pred_proba_test > threshold

print(classification_report(y_train, y_pred_train))
print(classification_report(y_test, y_pred_test))

training_cm = confusion_matrix(y_train, y_pred_train)
print(training_cm)
training_test_cm = confusion_matrix(y_test, y_pred_test)
print(test_cm)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    325551
           1       0.78      0.45      0.57       539

    accuracy                           1.00    326090
   macro avg       0.89      0.73      0.79    326090
weighted avg       1.00      1.00      1.00    326090

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    108446
           1       0.06      0.03      0.04       252

    accuracy                           1.00    108698
   macro avg       0.53      0.51      0.52    108698
weighted avg       1.00      1.00      1.00    108698

[[325482     69]
 [   295    244]]
[[107639    807]
 [   216     36]]


In [194]:
print(f'Training Savings: {(training_cm[1][1]*4000) - (training_cm[0][1]*500)}')
print(f'Training Test Savings: {(training_test_cm[1][1]*4000) - (training_test_cm[0][1]*500)}')

Training Savings: 941500
Training Test Savings: -29500


In [182]:
testing_df = pd.read_csv(
    filepath_or_buffer='testing_faults_diagnostics.csv',
    low_memory=False
)

In [183]:
testing_target = testing_df[target]

In [184]:
testing_df = testing_df[features]
testing_df.columns

Index(['spn', 'fmi', 'Severity_Level', 'BarometricPressure',
       'EngineCoolantTemperature', 'EngineLoad', 'EngineOilPressure',
       'EngineOilTemperature', 'EngineRpm', 'FuelRate', 'FuelTemperature',
       'IntakeManifoldTemperature', 'Speed', 'SwitchedBatteryVoltage',
       'Throttle', 'TurboBoostPressure'],
      dtype='object')

In [195]:
threshold = 0.5

y_pred_proba = pipe.predict_proba(testing_df)[:,1]

y_pred = y_pred_proba > threshold

print(classification_report(testing_target, y_pred))

testing_cm = confusion_matrix(testing_target, y_pred)
print(testing_cm)

              precision    recall  f1-score   support

           0       1.00      0.84      0.91     64808
           1       0.00      0.16      0.00       162

    accuracy                           0.83     64970
   macro avg       0.50      0.50      0.46     64970
weighted avg       1.00      0.83      0.91     64970

[[54171 10637]
 [  136    26]]


In [196]:
print(f'Savings: {(testing_cm[1][1]*4000) - (testing_cm[0][1]*500)}')

Savings: -5214500
